# Honest Assessment: When to Use PINNs

**Time: ~25 minutes**

You've built PINNs from scratch, trained them on ODEs and PDEs, used them for parametric problems and inverse inference. Now: an honest look at when they work, when they fail, and what to use instead.

## Where PINNs Excel

### 1. Inverse Problems

This is where PINNs genuinely outperform most alternatives. Inferring unknown parameters from sparse, noisy data — while enforcing physical consistency — is their strongest use case.

**Why they're good here:** The physics loss acts as an incredibly strong regularizer. Even with 10 noisy data points, the ODE/PDE constraint rules out physically impossible solutions.

**Real wins:**
- Raissi et al. (2019): infer Navier-Stokes parameters from velocity data, recover hidden pressure field
- Material property identification from sparse sensor measurements
- Hemodynamics: infer blood flow parameters from medical imaging

### 2. Multi-Physics and Complex Coupling

When you have multiple coupled PDEs and don't want to build a monolithic solver. The PINN approach is modular: just add more residual terms.

### 3. Mesh-Free Evaluation

Once trained, a PINN evaluates at any point instantly — no mesh interpolation, no grid dependency. This is valuable for:
- High-dimensional problems (curse of dimensionality for meshes)
- Complex, irregular geometries where meshing is painful
- When you need derivatives of the solution (free from autograd)

## Where PINNs Struggle

### 1. Spectral Bias

Neural networks learn low-frequency components first. High-frequency solutions (fast oscillations, sharp gradients) converge slowly or not at all.

**Mitigations:** Ansatz (embed known frequencies), Fourier features, adaptive activation functions. These help but don't eliminate the problem.

### 2. Sharp Gradients and Discontinuities

Shocks, contact discontinuities, phase boundaries — smooth activation functions can't represent these well. The PINN will either smear them out or oscillate.

### 3. Long Time Integration

PINNs trained on `t in [0, T]` tend to lose accuracy at large T. The physics residual is satisfied in an average sense, and errors accumulate. Time-stepping PINNs exist but add complexity.

### 4. Accuracy Guarantees

A PINN minimizes a loss function — it doesn't guarantee convergence to the true solution. Low loss ≠ low error. The residual can be small while the solution is wrong (especially near boundaries).

Classical solvers (FEM with a priori error bounds, spectral methods with exponential convergence) offer mathematical guarantees PINNs cannot.

### 5. Computational Cost for Standard Problems

For a well-posed 2D PDE on a simple domain, a classical solver will typically be 10-100x faster and more accurate than a PINN. The autograd overhead per training step is significant.

In [ ]:
# Quick demonstration: a PINN failure mode
# Try solving u'(t) = -100*u on [0, 1] — fast decay, stiff problem
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(1, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 1),
)

t_phys = torch.linspace(0, 1, 200).unsqueeze(1).requires_grad_(True)
t_ic = torch.zeros(1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

k = 100.0  # Stiff!
for epoch in range(10000):
    optimizer.zero_grad()
    u = model(t_phys)
    du = torch.autograd.grad(u, t_phys, torch.ones_like(u), create_graph=True)[0]
    loss = torch.mean((du + k * u)**2) + 50 * (model(t_ic) - 1)**2
    loss.backward()
    optimizer.step()

t_test = torch.linspace(0, 1, 500).unsqueeze(1)
with torch.no_grad():
    u_pred = model(t_test).numpy()
u_exact = np.exp(-k * t_test.numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(t_test.numpy(), u_exact, 'b-', label='Exact', linewidth=2)
axes[0].plot(t_test.numpy(), u_pred, 'r--', label='PINN', linewidth=2)
axes[0].set_xlabel('t'); axes[0].set_ylabel('u(t)')
axes[0].set_title(f"u' = -{k}u — PINN struggles with stiff problems")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(t_test.numpy()[:50], np.abs(u_pred[:50] - u_exact[:50]) + 1e-10)
axes[1].set_xlabel('t'); axes[1].set_ylabel('|error|')
axes[1].set_title('Error (early time)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Stiff problems need the network to resolve O(1/k) = {1/k} time scales.")
print("This is a well-known PINN limitation.")

## The Decision Framework

Ask these questions before choosing PINNs:

### 1. Is there a working classical solver?
- **Yes, and it's fast** → Use it. PINNs are unlikely to be better.
- **Yes, but it's slow/expensive** → PINNs might help for forward problems.
- **No** → PINNs are worth trying (complex geometry, high dimensions, multi-physics).

### 2. Is this an inverse problem?
- **Yes** → PINNs are a strong choice. The physics-data fusion is their sweet spot.
- **No** → Forward PINNs compete with classical methods (usually unfavorably).

### 3. Do you need a differentiable solution?
- **Yes** (for optimization, control, sensitivity) → PINNs provide this natively.
- **No** → Less of an advantage over classical methods.

### 4. Is the solution smooth?
- **Yes** → Good for PINNs.
- **No** (shocks, discontinuities) → PINNs will struggle.

### 5. Do you need accuracy guarantees?
- **Yes** (safety-critical, certification) → Use validated classical methods.
- **No** (research, exploration, screening) → PINNs are fine.

## Alternatives and Complements

| Method | Best for | Relationship to PINNs |
|--------|----------|----------------------|
| **FEM/FDM** | Standard forward problems | Better accuracy, worse for inverse |
| **Neural Operators** (DeepONet, FNO) | Learning solution operators from data | Data-hungry but generalise better |
| **Bayesian methods** | Uncertainty quantification | Can be combined with PINNs |
| **Reduced-order models** | Fast parametric solutions | Faster inference, less flexible |
| **Hybrid PINN + FEM** | Best of both worlds | FEM for coarse, PINN for refinement |

### Neural Operators vs PINNs

- **Neural Operators** (FNO, DeepONet): learn the solution operator from many (input, output) pairs. Need training data (from simulations or experiments). Generalize to new inputs at inference time.
- **PINNs**: learn a single solution from the equation itself. No training data needed. Must retrain for new problems (unless parametric).

Neural operators are better when you have plentiful data and need fast inference on many instances. PINNs are better when data is scarce and you have a good PDE model.

## Active Research Directions

PINNs are an active research area. Key open problems:

1. **Convergence theory** — when and why do PINNs converge? (Partially answered for simple cases)
2. **Adaptive collocation** — put more points where the residual is large
3. **Multi-scale problems** — handling features at different scales simultaneously
4. **Time-stepping PINNs** — solve long-time problems in windows
5. **Certified error bounds** — bridging the gap with classical methods
6. **Foundation models for PDEs** — pre-trained models fine-tuned for specific equations

## What You've Learned

Across these 8 notebooks, you've:

| Notebook | Key skill |
|----------|----------|
| 01 | Understand what PINNs are and aren't |
| 02 | Use `torch.autograd.grad` for exact derivatives |
| 03 | Build a complete PINN from scratch |
| 04 | Know when physics helps (sparse data, extrapolation) |
| 05 | Handle PDEs, BCs, and 2D collocation |
| 06 | Apply training tricks (weighting, Ansatz, scheduling) |
| 07 | Build parametric and inverse PINNs |
| 08 | Make informed decisions about when to use PINNs |

## Where to Go From Here

1. **Use the `pinn` library** in this repo to solve your own equations — see `libs/pinn/README.md`
2. **Read the experiments** — `experiments/` has 10 production-grade PINN implementations
3. **Add your own experiment** — follow `docs/adding_experiments.md`
4. **Read the papers**:
   - Raissi, Perdikaris, Karniadakis (2019). *Physics-informed neural networks.* J. Comput. Phys. 378.
   - Lu et al. (2021). *DeepXDE: A Deep Learning Library for Solving Differential Equations.*
   - Karniadakis et al. (2021). *Physics-informed machine learning.* Nature Rev. Phys. 3.